# 📊 Análisis de Personal y Costo de Planta 2025### Guía paso a paso, explicada de forma sencillaEste cuaderno explica, **sin necesidad de saber programar**, qué hace cada parte del proceso para analizar la base de datos de personal.Vamos a ver:1. Qué archivo usamos y por qué2. Cómo se "leen" los datos3. Cómo se unen dos tablas en una sola4. Cómo se hacen consultas y gráficos5. Cómo se exportan los resultados a Excel

## 🗂️ ¿Qué problema resolvemos?Tenemos un archivo de Excel (`.xlsb`) con información de **personal y costos de planta** de muchas entidades del Estado.Ese archivo tiene **varias hojas** (como pestañas en Excel), y la información que necesitamos está repartida en dos de ellas:- Una hoja con los **datos de cada cargo** (sueldo, tipo de vinculación, dependencia, etc.)- Otra hoja con el **nombre de cada entidad** (a qué departamento y ciudad pertenece)Para poder analizar todo junto, primero hay que **unir esas dos hojas**, como si pegáramos dos tablas de Excel usando una columna en común (el código de la entidad).

## 🔧 Paso 1: Preparar las herramientasAntes de trabajar con el archivo, necesitamos instalar unos "complementos" que permiten:- Leer archivos `.xlsb` (un tipo especial de Excel)- Crear gráficos- Guardar resultados en Excel normalEsto es como instalar una aplicación antes de usarla. Solo se hace **una vez** al iniciar.

In [ ]:
# Instalar las herramientas necesarias para trabajar con el archivo!pip install pyxlsb openpyxl matplotlib -q

## 📥 Paso 2: Leer el archivo y unir las dos hojasAquí pasan 3 cosas, en este orden:1. **Leemos** la hoja con los datos de personal (`CGR_PERSONAL_Y_COSTOS`)2. **Leemos** la hoja con los nombres de las entidades (`Entidades`)3. **Unimos** ambas usando el código de entidad como punto en común — así cada cargo queda asociado a su departamento y ciudad real> 💡 Es como un "Buscar V" (`VLOOKUP`) de Excel, pero hecho automáticamente para las 91,000+ filas a la vez.También optimizamos el uso de memoria de la computadora, para que el proceso no se "tranque" por la cantidad de datos.

In [ ]:
# Cargar y unir las hojas (CGR_PERSONAL_Y_COSTOS + Entidades)# con optimización de memoriaimport pandas as pdimport gcruta = "/content/Personal y Costo Planta 2025 06-04-2026-TOTAL_TOTAL1.xlsb"df = pd.read_excel(ruta, engine="pyxlsb", sheet_name="CGR_PERSONAL_Y_COSTOS")entidades = pd.read_excel(ruta, engine="pyxlsb", sheet_name="Entidades")df["CODENTIDAD"] = df["CODENTIDAD"].astype(str).str.strip()entidades["BK_CHIP"] = entidades["BK_CHIP"].astype(str).str.strip()df_completo = df.merge(    entidades[["BK_CHIP", "DS_ENTIDAD", "DS_DEPARTAMENTO", "DS_CIUDAD", "DS_AMBITO_NOMBRE_CGR_CHIP"]],    left_on="CODENTIDAD",    right_on="BK_CHIP",    how="left")del df, entidadesgc.collect()# Optimizar tipos: hacer los datos mas livianos para la memoriafor col in df_completo.select_dtypes(include="number").columns:    df_completo[col] = pd.to_numeric(df_completo[col], downcast="float")for col in df_completo.select_dtypes(include="object").columns:    if df_completo[col].nunique() / len(df_completo) < 0.5:        df_completo[col] = df_completo[col].astype("category")print(f"Filas y columnas finales: {df_completo.shape}")print(f"Registros sin nombre de entidad encontrado: {df_completo[\'DS_ENTIDAD\'].isna().sum()}")

## 👀 Paso 3: Ver qué información tenemos disponibleAntes de hacer preguntas a los datos, es bueno **ver la lista completa de columnas** disponibles.Esto es como mirar los encabezados de un Excel para saber qué se puede consultar: departamento, ciudad, tipo de vinculación, sueldo, etc.

In [ ]:
# Ver todas las columnas finales disponiblesfor i, c in enumerate(df_completo.columns):    tipo = "numérica (montos/cantidades)" if pd.api.types.is_numeric_dtype(df_completo[c]) else "texto/categoría"    valores_unicos = df_completo[c].nunique()    print(f"{i:2d}. {c}   -> {tipo}, {valores_unicos} valores distintos")

## 🧮 Paso 4: La "máquina de consultas"Aquí creamos dos herramientas que vamos a reutilizar todas las veces que queramos:**`ver_combinaciones(...)`**Antes de hacer una consulta grande, nos dice cuántos resultados distintos va a generar. Así evitamos pedir algo demasiado grande que tranque la computadora.**`consulta(...)`**Es la herramienta principal. Le decimos:- Por qué columnas agrupar (por ejemplo: departamento + tipo de vinculación)- Si queremos **contar** registros o **sumar** un valor (como el sueldo)Y ella automáticamente:1. Calcula la tabla de resultados2. Dibuja un gráfico3. Guarda todo en un archivo Excel

In [ ]:
# Herramienta 1: calcular cuántas combinaciones generaría una consultadef ver_combinaciones(group_cols):    for c in group_cols:        if c not in df_completo.columns:            raise ValueError(f"Columna no encontrada: {c}")    n = df_completo[list(group_cols)].drop_duplicates().shape[0]    print(f"La consulta {group_cols} generaría aproximadamente {n} combinaciones distintas.")    return n

In [ ]:
# Herramienta 2: la función principal de consulta (tabla + gráfico + Excel)import matplotlib.pyplot as pltimport textwrapdef consulta(group_cols, metrica="contar", top_n=20, nombre_archivo=None, max_combinaciones=5000):    """    group_cols: lista de columnas (1 a 5) para agrupar    metrica: "contar" para contar registros, o el nombre de una columna numérica para sumarla    top_n: cuántas filas mostrar en el gráfico    """    if not (1 <= len(group_cols) <= 5):        raise ValueError("Debes usar entre 1 y 5 columnas para agrupar")    for c in group_cols:        if c not in df_completo.columns:            raise ValueError(f"Columna no encontrada: {c}")    cols_necesarias = list(group_cols) + ([metrica] if metrica != "contar" else [])    subset = df_completo[cols_necesarias].copy()    for c in group_cols:        subset[c] = subset[c].astype(str)    if metrica == "contar":        resultado = subset.groupby(group_cols, dropna=False, observed=True).size().reset_index(name="Cantidad")        col_valor = "Cantidad"    else:        if metrica not in df_completo.columns:            raise ValueError(f"Columna de métrica no encontrada: {metrica}")        resultado = subset.groupby(group_cols, dropna=False, observed=True)[metrica].sum().reset_index()        col_valor = metrica    resultado = resultado.sort_values(col_valor, ascending=False).reset_index(drop=True)    del subset    gc.collect()    print(f"\n=== Consulta: agrupado por {group_cols} | métrica: {metrica} ===")    print(f"Total de combinaciones: {len(resultado)}")    print(resultado.head(30).to_string(index=False))    if len(resultado) > max_combinaciones:        print(f"\nHay {len(resultado)} combinaciones, demasiadas para graficar.")        print("Se omite el gráfico, pero la tabla completa sí se guarda en Excel.")    else:        grafico_data = resultado.head(top_n) if top_n else resultado        grafico_data = grafico_data.copy()        if len(group_cols) > 1:            etiqueta = grafico_data[group_cols[0]].astype(str)            for c in group_cols[1:]:                etiqueta = etiqueta + " | " + grafico_data[c].astype(str)        else:            etiqueta = grafico_data[group_cols[0]].astype(str)        etiqueta = etiqueta.apply(lambda x: "\n".join(textwrap.wrap(x, 40)))        usar_horizontal = len(grafico_data) > 8        plt.figure(figsize=(10, max(4, len(grafico_data) * 0.35)) if usar_horizontal else (10, 6))        if usar_horizontal:            plt.barh(etiqueta[::-1], grafico_data[col_valor][::-1])            plt.xlabel(col_valor)        else:            plt.bar(etiqueta, grafico_data[col_valor])            plt.ylabel(col_valor)            plt.xticks(rotation=45, ha="right")        plt.title(f"{col_valor} por {\' + \'.join(group_cols)}")        plt.tight_layout()        plt.show()        plt.close()    if nombre_archivo is None:        nombre_archivo = "consulta_" + "_".join(group_cols).replace(" ", "_")[:50]    ruta_salida = f"/content/{nombre_archivo}.xlsx"    resultado.to_excel(ruta_salida, index=False)    print(f"\nExcel guardado en: {ruta_salida}")    del resultado    gc.collect()    return pd.read_excel(ruta_salida)

## ❓ Paso 5: Hacer preguntas a los datosAhora viene la parte más útil: **preguntarle cosas a los datos** usando la función `consulta(...)`.Cada pregunta sigue el mismo patrón sencillo:> "Quiero ver [qué medir], agrupado por [una o varias columnas]"Por ejemplo:- ¿Cuántos cargos hay por departamento?- ¿Cuántos tipos de vinculación hay en cada departamento?- ¿Cuánto suman los sueldos básicos por dependencia?Veamos ejemplos reales.

### Ejemplo 1: ¿Cuántos cargos hay por departamento?Aquí solo contamos registros, agrupando por una sola columna.

In [ ]:
res1 = consulta(["DS_DEPARTAMENTO"], metrica="contar")

### Ejemplo 2: Tipo de vinculación dentro de cada departamentoAquí agrupamos por **dos** columnas a la vez.

In [ ]:
res2 = consulta(["DS_DEPARTAMENTO", "TIPO DE VINCULACIÓN"], metrica="contar")

### Ejemplo 3: Suma de sueldos por dependenciaAquí, en vez de **contar**, **sumamos** una columna de dinero (la asignación básica anual).

In [ ]:
res3 = consulta(    ["DS_DEPARTAMENTO", "UNIDAD EJECUTORA ó DEPENDENCIA"],    metrica="ASIGNACIÓN BÁSICA ANUAL")

### Ejemplo 4: Consulta más detallada (hasta 5 columnas)Antes de pedir consultas muy detalladas, conviene revisar primero cuántos resultados va a traer, con `ver_combinaciones(...)`.

In [ ]:
ver_combinaciones(["DS_DEPARTAMENTO", "DS_CIUDAD", "TIPO DE VINCULACIÓN", "DENOMINACIÓN DEL CARGO", "GRADO"])

In [ ]:
res4 = consulta(    ["DS_DEPARTAMENTO", "DS_CIUDAD", "TIPO DE VINCULACIÓN", "DENOMINACIÓN DEL CARGO", "GRADO"],    metrica="contar",    top_n=15)

## 💾 Paso 6: Descargar los resultadosCada vez que hacemos una consulta, automáticamente se guarda un archivo Excel en la carpeta del cuaderno.Para bajarlo a tu computadora, solo escribe el nombre del archivo que quieres descargar.

In [ ]:
from google.colab import files# Reemplaza el nombre por el archivo que quieras descargar:# files.download("/content/consulta_DS_DEPARTAMENTO.xlsx")

## ✅ Resumen: ¿qué logramos?1. Unimos dos hojas del Excel en una sola tabla completa2. Vimos qué columnas están disponibles para consultar3. Creamos una herramienta reutilizable de consultas4. Hicimos preguntas a los datos: conteos y sumas, con gráficos automáticos5. Exportamos cada resultado a Excel para compartirlo### 🔁 ¿Cómo sigo usando esto?Solo necesitas escribir una línea como esta, cambiando las columnas que te interesen:```pythonconsulta(["MI_COLUMNA_1", "MI_COLUMNA_2"], metrica="contar")```¡Y listo! Tabla, gráfico y Excel automáticos.